# Brain Extraction (BET)

Almost every dMRI step requires a **brain mask** — a binary image that tells the software which voxels are brain and which are skull, scalp, and background. Without a good mask:

- DTI fitting wastes time on non-brain voxels and produces nonsense values there
- Eddy correction can be misled by skull signal
- Tractography seeds outside the brain
- Registration fails because the skull dominates the cost function

## The three approaches

| Tool | Method | Input | Notes |
|---|---|---|---|
| **FSL BET** | Deformable surface model | b=0 or T1w | Most widely used; `–f` threshold is critical |
| **MRtrix3 dwi2mask** | dMRI-specific (b=0 + DWI signal) | Full DWI | Understands diffusion contrast better than BET |
| **DIPY median_otsu** | Otsu threshold on b=0 median | b=0 volumes | Pure Python; no external tool needed |

> **Critical parameter — FSL BET `-f` (fractional intensity threshold):**  
> `-f 0.5` is the default. Lower values (e.g. `-f 0.2`) include more tissue (less aggressive).  
> Higher values (e.g. `-f 0.7`) exclude more (more aggressive).  
> **For dMRI, `-f 0.2` to `-f 0.35` is usually better** — the b=0 image has lower contrast than T1w and BET tends to over-strip.

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import run

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
prep_dir.mkdir(parents=True, exist_ok=True)

dwi_nii = str(data_dir / 'data.nii.gz')
bvals_f = str(data_dir / 'bvals')

# Extract b=0 mean image (input to BET)
bvals = np.loadtxt(bvals_f)
img   = nib.load(dwi_nii)
data  = img.get_fdata()

b0_vols  = data[..., bvals < 50]
b0_mean  = b0_vols.mean(axis=-1).astype(np.float32)
b0_path  = str(prep_dir / 'b0_mean.nii.gz')
nib.save(nib.Nifti1Image(b0_mean, img.affine, img.header), b0_path)
print(f'b=0 mean image saved: {b0_path}  (shape {b0_mean.shape})')

## Approach A: FSL BET

In [ ]:
# ─── [FSL] bet ────────────────────────────────────────────────────────────────
#
# -f 0.25 : fractional intensity threshold (lower = keep more tissue)
# -g 0    : vertical gradient in f (0 = uniform)
# -m      : output binary brain mask
# -R      : robust brain centre estimation (more iterations; recommended for dMRI)

fsl_mask_base = str(prep_dir / 'bet_brain')

fsl_bet_cmd = [
    'bet', b0_path, fsl_mask_base,
    '-f', '0.25',
    '-g', '0',
    '-m',          # write mask as <base>_mask.nii.gz
    '-R',          # robust centre estimation
]
print('[FSL] BET command:')
print(' '.join(fsl_bet_cmd))

result = subprocess.run(fsl_bet_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    mask_path = fsl_mask_base + '_mask.nii.gz'
    mask_vols = nib.load(mask_path).get_fdata().sum()
    print(f'✓ FSL BET mask: {int(mask_vols)} brain voxels')

## Approach B: MRtrix3 dwi2mask

In [ ]:
# ─── [MRtrix3] dwi2mask ───────────────────────────────────────────────────────
#
# dwi2mask uses the full DWI signal contrast — not just b=0.
# It is generally more robust for dMRI than BET because it understands
# that high-b images look different from b=0.
#
# Requires the .mif file with embedded gradient table from the intro notebook.

mrt_mif   = str(prep_dir / 'dwi_raw.mif')
mrt_mask  = str(prep_dir / 'mask_mrtrix.nii.gz')

# First convert to .mif if not done yet
if not Path(mrt_mif).exists():
    subprocess.run([
        'mrconvert', dwi_nii, mrt_mif,
        '-fslgrad', str(data_dir / 'bvecs'), bvals_f,
        '-force'
    ], capture_output=True)

mrt_mask_cmd = [
    'dwi2mask', 'legacy',   # 'legacy' = original algorithm; also try 'fslbet'
    mrt_mif,
    mrt_mask,
    '-force',
]
print('[MRtrix3] dwi2mask command:')
print(' '.join(mrt_mask_cmd))

result = subprocess.run(mrt_mask_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    mask_vols = nib.load(mrt_mask).get_fdata().sum()
    print(f'✓ MRtrix3 mask: {int(mask_vols)} brain voxels')

## Approach C: DIPY median_otsu

In [ ]:
# ─── [DIPY] median_otsu ───────────────────────────────────────────────────────
#
# median_otsu:
#   1. Takes the median across b=0 volumes
#   2. Applies a median filter to smooth
#   3. Applies Otsu thresholding
#   4. Fills holes and removes small objects
#
# vol_idx : indices of b=0 volumes to use
# median_radius : size of the median filter kernel (default 4)
# numpass : number of dilation passes (default 4)

from dipy.segment.mask import median_otsu

b0_indices = np.where(bvals < 50)[0].tolist()

print(f'Using {len(b0_indices)} b=0 volumes: {b0_indices}')

b0_masked, dipy_mask = median_otsu(
    data,
    vol_idx=b0_indices,
    median_radius=2,
    numpass=1,
    autocrop=False,
    dilate=1,
)

dipy_mask_path = str(prep_dir / 'mask_dipy.nii.gz')
nib.save(nib.Nifti1Image(dipy_mask.astype(np.uint8), img.affine), dipy_mask_path)

print(f'✓ DIPY mask: {dipy_mask.sum()} brain voxels')

## Compare all three masks

In [ ]:
# Overlay each mask on the b=0 mean image and compare
masks = {}
for label, path in [
    ('FSL BET',   fsl_mask_base + '_mask.nii.gz'),
    ('MRtrix3',   mrt_mask),
    ('DIPY',      dipy_mask_path),
]:
    if Path(path).exists():
        masks[label] = nib.load(path).get_fdata().astype(bool)

# Always have DIPY
if 'DIPY' not in masks:
    masks['DIPY'] = dipy_mask

z = b0_mean.shape[2] // 2
n = len(masks)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, (label, mask_vol) in zip(axes, masks.items()):
    # Show b=0 as background
    ax.imshow(b0_mean[:, :, z].T, cmap='gray', origin='lower')
    # Overlay mask edge in red
    from scipy.ndimage import binary_erosion
    edge = mask_vol[:, :, z] & ~binary_erosion(mask_vol[:, :, z])
    overlay = np.zeros((*edge.shape, 4))
    overlay[edge] = [1, 0, 0, 0.9]   # red edge
    ax.imshow(overlay.transpose(1, 0, 2), origin='lower')
    ax.set_title(f'{label}\n({int(mask_vol.sum())} voxels)', fontsize=11)
    ax.axis('off')

fig.suptitle('Brain masks — red outline shows mask boundary\n'
             'Check: no skull included, no brain cut off',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: Dice overlap between masks
labels = list(masks.keys())
n = len(labels)

def dice(a, b):
    return 2 * (a & b).sum() / (a.sum() + b.sum() + 1e-10)

print('=== Dice similarity between masks ===')
print('(1.0 = identical, <0.95 = meaningful difference worth investigating)')
print()
for i in range(n):
    for j in range(i+1, n):
        d = dice(masks[labels[i]], masks[labels[j]])
        print(f'  {labels[i]:<12} vs {labels[j]:<12}: Dice = {d:.3f}')

print()
print('Which mask to use downstream?')
print('  • If Dice > 0.97 between all: any mask is fine')
print('  • If Dice < 0.95: visually inspect and pick the most complete')
print('  • DIPY is the safe default when FSL/MRtrix3 not available')

## Troubleshooting poor masks

| Problem | What you see | Fix |
|---|---|---|
| Over-stripping | Brain tissue cut at edges | Lower `-f` (e.g. 0.2) or use `-R` |
| Under-stripping | Skull/scalp included | Raise `-f` (e.g. 0.4) |
| Holes in mask | Islands of zero inside brain | DIPY: increase `numpass`; FSL: add `-S` |
| Wrong centre | BET finds wrong initial centre | Use `-c x y z` to specify manually |

**Next**: [Denoising →](01_denoising.ipynb)